In [2]:
import pandas as pd
import numpy as np
import random 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings('ignore')


# ----------------------------------------------------------
# 운명의 길 한국어 매핑
# ----------------------------------------------------------
PATH_KO = {
    'Nihility'     : '공허',
    'Destruction'  : '파멸',
    'Hunt'         : '수렵',
    'Erudition'    : '지식',
    'Harmony'      : '화합',
    'Abundance'    : '풍요',
    'Preservation' : '보존',
    'Remembrance'  : '기억',
    'Elation'      : '환락',
}

FEATURE_KO = {
    'atk_base'   : '공격력',
    'def_base'   : '방어력',
    'hp_base'    : 'HP',
    'base_spd'   : '속도',
    'base_aggro' : '어그로 수치',
    'max_energy' : '최대 에너지',
}

# ----------------------------------------------------------
# 데이터셋 로드
# ----------------------------------------------------------

chars = pd.read_csv("characters.csv")
stats = pd.read_csv("character_stats.csv")

stats_max = stats[stats['ascension'] == 6].copy()
df = chars.merge(stats_max, on='character_id')
df['path_ko'] = df['path'].map(PATH_KO)

print(f"총 캐릭터 수 : {len(df)}명")
print(f"운명의 길    : {' / '.join(PATH_KO[p] for p in sorted(PATH_KO))}")
print()

# ----------------------------------------------------------
# 피처 & 타겟 정의
# ----------------------------------------------------------
features = list(FEATURE_KO.keys())

X = df[features].fillna(df[features].median())
y = df['path_ko']

print("[ 사용 피처 ]")
for eng, ko in FEATURE_KO.items():
    print(f"  - {ko} ({eng})")

print(f"\n[ 운명의 길별 캐릭터 수 ]")
for path, cnt in y.value_counts().items():
    print(f"  {path:6s} : {cnt}명")
print()

# ----------------------------------------------------------
# 학습 / 테스트 데이터 분리 (80:20)
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ----------------------------------------------------------
# 모델 학습
# ----------------------------------------------------------

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)


svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train_scaled, y_train)


# ----------------------------------------------------------
# 게임 밸런스
# ----------------------------------------------------------
y_pred_clf = clf.predict(X_test)
y_pred_svm = svm.predict(X_test_scaled)
acc_clf    = accuracy_score(y_test, y_pred_clf)
acc_svm    = accuracy_score(y_test, y_pred_svm)
best_acc  = max(acc_clf, acc_svm)
best_pred = y_pred_clf if acc_clf >= acc_svm else y_pred_svm
best_acc  = max(acc_clf, acc_svm)
best_name = "분류 모델" if acc_clf >= acc_svm else "보조 모델"

print()
print("=" * 60)
print("게임 밸런스")
print("=" * 60)
print()
print(f"  정확도 {best_acc*100:.1f}% : 스탯만으로 운명의 길을 분류한 결과")

print("  [ 오분류 캐릭터 목록 ]")
result_df = pd.DataFrame({
    '캐릭터'    : df.loc[X_test.index, 'character_name'].values,
    '실제 운명' : y_test.values,
    '예측 운명' : best_pred,
})
wrong = result_df[result_df['실제 운명'] != result_df['예측 운명']]
if len(wrong) > 0:
    for _, row in wrong.iterrows():
        print(f"  ✗ {row['캐릭터']:25s}  실제: {row['실제 운명']:4s} → 예측: {row['예측 운명']}")
else:
    print("모든 테스트 샘플이 정확히 분류되었습니다.")

importances = clf.feature_importances_
feat_df = pd.DataFrame({
    '피처'  : [FEATURE_KO[f] for f in features],
    '중요도' : importances
}).sort_values('중요도', ascending=False)

print()
print("  [ 스탯 기여도 순위 ]")
for _, row in feat_df.iterrows():
    bar = "█" * int(row['중요도'] * 40)
    print(f"  {row['피처']:10s}  {bar} {row['중요도']:.4f}")




총 캐릭터 수 : 86명
운명의 길    : 풍요 / 파멸 / 환락 / 지식 / 화합 / 수렵 / 공허 / 보존 / 기억

[ 사용 피처 ]
  - 공격력 (atk_base)
  - 방어력 (def_base)
  - HP (hp_base)
  - 속도 (base_spd)
  - 어그로 수치 (base_aggro)
  - 최대 에너지 (max_energy)

[ 운명의 길별 캐릭터 수 ]
  공허     : 14명
  파멸     : 14명
  수렵     : 12명
  화합     : 12명
  지식     : 10명
  풍요     : 7명
  기억     : 6명
  보존     : 6명
  환락     : 5명


게임 밸런스

  정확도 77.8% : 스탯만으로 운명의 길을 분류한 결과
  [ 오분류 캐릭터 목록 ]
  ✗ The Dahlia                 실제: 공허   → 예측: 화합
  ✗ Asta                       실제: 화합   → 예측: 공허
  ✗ Silver Wolf LV.999         실제: 환락   → 예측: 화합
  ✗ Cyrene                     실제: 기억   → 예측: 화합

  [ 스탯 기여도 순위 ]
  어그로 수치      ██████████ 0.2736
  공격력         ██████ 0.1679
  속도          ██████ 0.1524
  최대 에너지      █████ 0.1488
  방어력         █████ 0.1382
  HP          ████ 0.1190


In [2]:
print("=" * 60)
print("예측 및 정확도")
print("=" * 60)

print(f"  [분류 모델]  정확도 : {acc_clf:.4f}  ({acc_clf*100:.1f}%)")
print(f"  [보조 모델]  정확도 : {acc_svm:.4f}  ({acc_svm*100:.1f}%)")
print()


print(f"  ★ 최종 채택 : {best_name} (정확도 {best_acc*100:.1f}%)")
print()

print("[ 운명의 길별 상세 분류 결과 ]")
print(classification_report(y_test, y_pred_clf,
                             labels=sorted(PATH_KO.values()),
                             zero_division=0))


예측 및 정확도
  [분류 모델]  정확도 : 0.6667  (66.7%)
  [보조 모델]  정확도 : 0.7778  (77.8%)

  ★ 최종 채택 : 보조 모델 (정확도 77.8%)

[ 운명의 길별 상세 분류 결과 ]
              precision    recall  f1-score   support

          공허       0.50      0.67      0.57         3
          기억       0.00      0.00      0.00         1
          보존       1.00      1.00      1.00         1
          수렵       0.75      1.00      0.86         3
          지식       1.00      0.50      0.67         2
          파멸       1.00      1.00      1.00         3
          풍요       0.50      1.00      0.67         1
          화합       0.33      0.33      0.33         3
          환락       0.00      0.00      0.00         1

    accuracy                           0.67        18
   macro avg       0.56      0.61      0.57        18
weighted avg       0.62      0.67      0.63        18



In [ ]:
# ----------------------------------------------------------
# 8. 랜덤 캐릭터 예측 테스트
# ----------------------------------------------------------
print()
print("=" * 60)
print("랜덤 캐릭터 예측 테스트")
print("=" * 60)
print("  학습에 사용한 전체 데이터에서 캐릭터를 무작위로 뽑아")
print("  스탯만 보고 운명의 길을 맞히는지 확인합니다.")
print()

# 50명 무작위 추출
sample_indices = random.sample(range(len(df)), 50)
correct = 0

for i, idx in enumerate(sample_indices, 1):
    row         = df.iloc[idx]
    name        = row['character_name']
    actual_path = row['path_ko']
    stat_input  = pd.DataFrame([row[features].fillna(df[features].median().iloc[0])])

    predicted_path = clf.predict(stat_input)[0]
    is_correct     = "정답" if predicted_path == actual_path else "오답"
    if predicted_path == actual_path:
        correct += 1

    print(f"  [{i}] {name}")
    print(f"       공격력 {row['atk_base']:.0f}  |  방어력 {row['def_base']:.0f}  |  HP {row['hp_base']:.0f}")
    print(f"       속도 {row['base_spd']:.0f}  |  어그로 {row['base_aggro']:.0f}  |  최대에너지 {row['max_energy']:.0f}")
    print(f"       → 예측: {predicted_path:4s}   실제: {actual_path:4s}   {is_correct}")
    print()

print(f"  랜덤 테스트 결과 : {correct}/{len(sample_indices)} 정답 ({correct/len(sample_indices)*100:.0f}%)")


랜덤 캐릭터 예측 테스트
  학습에 사용한 전체 데이터에서 캐릭터를 무작위로 뽑아
  스탯만 보고 운명의 길을 맞히는지 확인합니다.

  [1] Sparkle
       공격력 242  |  방어력 224  |  HP 646
       속도 101  |  어그로 100  |  최대에너지 110
       → 예측: 화합     실제: 화합     정답

  [2] Archer
       공격력 287  |  방어력 224  |  HP 539
       속도 105  |  어그로 75  |  최대에너지 220
       → 예측: 수렵     실제: 수렵     정답

  [3] Natasha
       공격력 220  |  방어력 235  |  HP 539
       속도 98  |  어그로 100  |  최대에너지 90
       → 예측: 풍요     실제: 풍요     정답

  [4] Moze
       공격력 277  |  방어력 163  |  HP 375
       속도 111  |  어그로 75  |  최대에너지 120
       → 예측: 수렵     실제: 수렵     정답

  [5] Tribbie
       공격력 242  |  방어력 337  |  HP 485
       속도 96  |  어그로 100  |  최대에너지 120
       → 예측: 화합     실제: 화합     정답

  [6] Guinaifen
       공격력 269  |  방어력 204  |  HP 408
       속도 106  |  어그로 100  |  최대에너지 120
       → 예측: 공허     실제: 공허     정답

  [7] Tingyun
       공격력 245  |  방어력 184  |  HP 392
       속도 112  |  어그로 100  |  최대에너지 130
       → 예측: 화합     실제: 화합     정답

  [8] Acheron
       공격력 323  |  방어력 202  

In [71]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler as KNNScaler

# 각 스탯의 실제 min~max 범위 안에서 랜덤 생성
stat_ranges = {
    'atk_base'   : (int(X['atk_base'].min()),   int(X['atk_base'].max())),
    'def_base'   : (int(X['def_base'].min()),   int(X['def_base'].max())),
    'hp_base'    : (int(X['hp_base'].min()),    int(X['hp_base'].max())),
    'base_spd'   : (int(X['base_spd'].min()),   int(X['base_spd'].max())),
    'base_aggro' : (int(X['base_aggro'].min()), int(X['base_aggro'].max())),
    'max_energy' : (int(X['max_energy'].min()), int(X['max_energy'].max())),
}

random_stats = {stat: random.randint(low, high) for stat, (low, high) in stat_ranges.items()}
random_input = pd.DataFrame([random_stats])

# 운명의 길 예측
predicted = clf.predict(random_input)[0]

print()
print("  [ 생성된 랜덤 캐릭터 스탯 ]")
for eng, ko in FEATURE_KO.items():
    print(f"  {ko:8s} : {random_stats[eng]}")
print()
print(f"  ★ 예측된 운명의 길 : {predicted}")
print()

# KNN으로 가장 가까운 캐릭터 3명 탐색
knn_scaler = KNNScaler()
X_knn = knn_scaler.fit_transform(X)
random_knn = knn_scaler.transform(random_input)

knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_knn, y)

distances, indices = knn.kneighbors(random_knn)

print("  [ 스탯이 가장 유사한 캐릭터 TOP 10 ]")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    similar = df.iloc[idx]
    print(f"  {rank}위  {similar['character_name']:25s}  운명의 길: {similar['path_ko']:4s}  (유사도 거리: {dist:.2f})")

print()
print("  [ 주변 10명의 운명의 길 분포 ]")
path_counts = {}
for idx in indices[0]:
    path = df.iloc[idx]['path_ko']
    path_counts[path] = path_counts.get(path, 0) + 1

total = sum(path_counts.values())
for path, count in sorted(path_counts.items(), key=lambda x: -x[1]):
    bar        = "█" * count
    ratio      = count / total * 100
    marker     = "  ← 예측 결과" if path == predicted else ""
    print(f"  {path:4s}  {bar:<10s} {count}명  ({ratio:.0f}%){marker}")

predicted_ratio = path_counts.get(predicted, 0) / total * 100
print()
if predicted_ratio >= 50:
    print(f"예측 신뢰도 높음 — 주변 캐릭터의 {predicted_ratio:.0f}%가 {predicted}")
elif predicted_ratio >= 30:
    print(f"예측 신뢰도 보통 — 주변 캐릭터의 {predicted_ratio:.0f}%가 {predicted}, 경계선에 가까운 스탯")
else:
    print(f"예측 신뢰도 낮음 — 주변 캐릭터의 {predicted_ratio:.0f}%만 {predicted}, 스탯만으로 역할 구분 어려움")


  [ 생성된 랜덤 캐릭터 스탯 ]
  공격력      : 323
  방어력      : 234
  HP       : 622
  속도       : 102
  어그로 수치   : 92
  최대 에너지   : 56

  ★ 예측된 운명의 길 : 공허

  [ 스탯이 가장 유사한 캐릭터 TOP 10 ]
  1위  Lingsha                    운명의 길: 풍요    (유사도 거리: 1.41)
  2위  Jiaoqiu                    운명의 길: 공허    (유사도 거리: 1.60)
  3위  Acheron                    운명의 길: 공허    (유사도 거리: 1.64)
  4위  Cerydra                    운명의 길: 화합    (유사도 거리: 1.65)
  5위  Hysilens                   운명의 길: 공허    (유사도 거리: 1.68)
  6위  Luocha                     운명의 길: 풍요    (유사도 거리: 1.75)
  7위  Robin                      운명의 길: 화합    (유사도 거리: 1.84)
  8위  Jing Yuan                  운명의 길: 지식    (유사도 거리: 1.85)
  9위  Welt                       운명의 길: 공허    (유사도 거리: 1.87)
  10위  Black Swan                 운명의 길: 공허    (유사도 거리: 1.87)

  [ 주변 10명의 운명의 길 분포 ]
  공허    █████      5명  (50%)  ← 예측 결과
  풍요    ██         2명  (20%)
  화합    ██         2명  (20%)
  지식    █          1명  (10%)

예측 신뢰도 높음 — 주변 캐릭터의 50%가 공허
